# 🌍 ISOM 260: Agents Meet the Real World (Gemini Version)

**Session 5 — From Simulated to Real** | Suffolk University | Prof. Hasan Arslan

---

## From Fake Data to Real APIs

In Session 4, you built an AI agent with **simulated tools** — a calculator (real math, but simple), stock prices (hardcoded dictionaries), and company info (static data). That was great for learning the pattern:

```
AI Agent = LLM + Tools + Reasoning Loop
```

But there was a problem: **the data never changed**. Apple's stock price was always $242.58. The company info was always the same. And if something went wrong? Your agent had no idea how to handle it.

### Today: The Real World

In this session, we upgrade our agents with:

1. 🌐 **Real APIs** — Wikipedia search, live web page fetching
2. 🛡️ **Error handling** — What happens when APIs fail, pages don't exist, or connections time out?
3. 💾 **Conversation memory** — Agents that remember what you said earlier

### The Session 4 → 5 → 8 Progression

| Session | Focus | Tools |
|---|---|---|
| **Session 4** | Build your first agent | Simulated (calculator, fake stocks) |
| **Session 5** (today) | Connect to the real world | Real APIs + error handling + memory |
| **Session 8** | Production agents | NanoClaw architecture deep-dive |

### 💡 Real-World Context: NanoClaw

Remember NanoClaw from Session 4? The 500-line agent that Andrej Karpathy praised? One of the things that makes it production-ready (unlike a classroom demo) is that it handles **real APIs**, **real errors**, and **real conversations**. Today you'll learn exactly how that works.

---

## 🚀 Part 1: Setup (2 minutes)

Install dependencies and connect to Gemini.

In [ ]:
# Install required packages
# google-genai: Gemini SDK
# beautifulsoup4: HTML parsing for web fetcher
# requests: HTTP requests for real APIs
!pip install -q google-genai beautifulsoup4 requests

In [ ]:
# Set your API key
# Get yours at: https://aistudio.google.com/apikey
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Option 2: Paste directly (less secure, but works for class)
    GOOGLE_API_KEY = "your-api-key-here"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

In [ ]:
# Verify connection
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say 'Agent ready!' in exactly 2 words."
)

print(response.text)
print(f"\n✅ Connection successful! Using model: gemini-2.5-flash")

---

## 🧮 Part 2: Bring Back the Calculator

Before we add real-world tools, let's bring back our calculator from Session 4. This gives us a known-good tool to combine with the new ones.

In [ ]:
# ============================================
# Calculator Tool — carried over from Session 4
# ============================================

from google.genai import types
import json

calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise mathematical calculator. Use this tool whenever you need to "
        "perform arithmetic calculations. It handles addition, subtraction, "
        "multiplication, division, and exponentiation with perfect accuracy. "
        "Always prefer this tool over mental math for any calculation."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The mathematical expression to evaluate, e.g. '2 + 2' or '(100 * 0.15) + 50'"
            }
        },
        "required": ["expression"]
    }
)

print("📝 Tool defined: calculator")

In [ ]:
# ============================================
# Calculator function — same as Session 4
# ============================================

def calculator(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.
    Returns the result as a string.
    """
    try:
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return f"Error: Expression contains invalid characters. Only numbers and +-*/.() are allowed."
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

# Quick test
print(f"2 + 2 = {calculator('2 + 2')}")
print(f"7394 * 8261 = {calculator('7394 * 8261')}")
print("\n✅ Calculator ready!")

In [ ]:
# ============================================
# Agent Loop — same pattern from Session 4
# ============================================
# The HEART of every AI agent: the loop that connects
# Gemini's thinking to real-world tool execution

def run_agent(user_message: str, tool_declarations: list, tool_functions: dict, verbose=True):
    """
    Run an AI agent that can use tools to answer questions.

    This is the same core pattern used by NanoClaw, OpenClaw,
    Claude Code, and every AI agent in production today.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"🗣️  User: {user_message}")
        print(f"{'='*60}")

    # Build the conversation
    contents = [
        types.Content(role="user", parts=[types.Part(text=user_message)])
    ]

    config = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=tool_declarations)],
        system_instruction="You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer. Always show your reasoning."
    )

    step = 0

    while True:
        step += 1

        # 📡 Send message to Gemini (with tool definitions)
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            config=config,
            contents=contents,
        )

        if verbose:
            print(f"\n🔄 Step {step}")

        # Get the model's response and add to history
        model_content = response.candidates[0].content
        contents.append(model_content)

        # Check if Gemini wants to use a tool
        function_calls = [p for p in model_content.parts if p.function_call]

        if not function_calls:
            # 🎉 No function calls — Gemini is done
            final_answer = response.text
            if verbose:
                print(f"\n🤖 Agent Answer:\n{final_answer}")
                print(f"\n✅ Done in {step} step(s)")
            return final_answer

        # Print any thinking text alongside function calls
        for part in model_content.parts:
            if not part.function_call and part.text:
                if verbose:
                    text = part.text
                    print(f"   🧠 Thinking: {text[:150]}..." if len(text) > 150 else f"   🧠 Thinking: {text}")

        # 🚀 Execute each function call
        function_response_parts = []
        for fc_part in function_calls:
            tool_name = fc_part.function_call.name
            tool_args = dict(fc_part.function_call.args)

            if verbose:
                print(f"   🛠️  Calling tool: {tool_name}")
                print(f"      Input: {json.dumps(tool_args, default=str)}")

            # Execute the tool!
            if tool_name in tool_functions:
                result = tool_functions[tool_name](**tool_args)
            else:
                result = f"Error: Unknown tool '{tool_name}'"

            if verbose:
                print(f"      Result: {result[:200]}..." if len(result) > 200 else f"      Result: {result}")

            function_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={"result": result}
                )
            )

        # Send function results back to Gemini
        contents.append(
            types.Content(role="user", parts=function_response_parts)
        )

        # Safety: prevent infinite loops
        if step > 10:
            return "Error: Agent exceeded maximum steps."

print("✅ Agent loop defined!")

In [ ]:
# ============================================
# 🎯 Quick test — make sure calculator still works
# ============================================

tools = [calculator_tool]
tool_functions = {"calculator": calculator}

run_agent("What is 2026 minus 1906? And what's that number squared?", tools, tool_functions)

---

## 🌐 Part 3: Wikipedia Search Tool (REAL API!)

Here's where things get exciting. Instead of hardcoded data, we're going to call the **real Wikipedia API**. This means:

- ✅ **Live data** — always up to date
- ✅ **Any topic** — not limited to a predefined dictionary
- ❗ **Can fail** — network errors, pages not found, timeouts

This is exactly the kind of tool that real agents like NanoClaw use. Let's build it!

### How the Wikipedia API works

```
GET https://en.wikipedia.org/api/rest_v1/page/summary/{topic}
→ Returns: title, extract (summary), URL
→ 404 if page doesn't exist
```

No API key needed! It's free and open.

In [ ]:
# ============================================
# 🌐 Wikipedia Search Tool — REAL API!
# ============================================

import requests

wikipedia_tool = types.FunctionDeclaration(
    name="wikipedia_search",
    description=(
        "Search Wikipedia for information about any topic. Returns a summary "
        "of the Wikipedia article including the title, a text extract, and "
        "a link to the full article. Use this whenever you need factual "
        "information about people, places, events, concepts, or organizations."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The topic to search for on Wikipedia, e.g. 'Artificial intelligence' or 'Suffolk University'"
            }
        },
        "required": ["topic"]
    }
)


def wikipedia_search(topic: str) -> str:
    """
    Search Wikipedia for a topic and return a summary.
    This calls the REAL Wikipedia API — live data!
    """
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic}"

    try:
        response = requests.get(
            url,
            timeout=10,
            headers={"User-Agent": "ISOM260-Agent/1.0 (Suffolk University classroom use)"}
        )

        if response.status_code == 404:
            return json.dumps({"error": f"No Wikipedia page found for '{topic}'. Try a different search term."})

        response.raise_for_status()
        data = response.json()

        return json.dumps({
            "title": data.get("title", ""),
            "summary": data.get("extract", ""),
            "url": data.get("content_urls", {}).get("desktop", {}).get("page", "")
        })

    except requests.exceptions.Timeout:
        return json.dumps({"error": "Wikipedia request timed out. Try again."})
    except requests.exceptions.RequestException as e:
        return json.dumps({"error": f"Wikipedia request failed: {str(e)}"})

print("📝 Tool defined: wikipedia_search")
print("   🌐 This calls the REAL Wikipedia API!")

In [ ]:
# ============================================
# Test the Wikipedia function directly (no agent)
# ============================================

# This shows you what the raw API returns
result = wikipedia_search("Artificial intelligence")
data = json.loads(result)

print(f"📖 Title: {data['title']}")
print(f"\n📝 Summary: {data['summary'][:300]}...")
print(f"\n🔗 URL: {data['url']}")
print(f"\n✅ Real Wikipedia data! This is LIVE — not hardcoded.")

In [ ]:
# ============================================
# 🎯 Test Wikipedia through the agent
# ============================================

tools = [calculator_tool, wikipedia_tool]
tool_functions = {"calculator": calculator, "wikipedia_search": wikipedia_search}

run_agent(
    "What is artificial intelligence? Give me a brief summary.",
    tools, tool_functions
)

---

## 🕸️ Part 4: Web Page Fetcher Tool

Wikipedia is great, but what about the rest of the internet? Let's build a tool that can **fetch and read any web page**. The agent can then use it to:

- Read news articles
- Check product pages
- Gather information from any public website

We'll use `requests` to fetch the HTML and `BeautifulSoup` to extract clean text.

In [ ]:
# ============================================
# 🕸️ Web Page Fetcher Tool — read any public URL
# ============================================

from bs4 import BeautifulSoup

web_fetch_tool = types.FunctionDeclaration(
    name="web_fetch",
    description=(
        "Fetch and read the text content of any public web page. "
        "Provide a full URL (starting with http:// or https://). "
        "Returns the main text content of the page, stripped of HTML tags. "
        "Use this when you need information from a specific website or URL."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The full URL to fetch, e.g. 'https://example.com'"
            }
        },
        "required": ["url"]
    }
)


def web_fetch(url: str) -> str:
    """
    Fetch a web page and extract its text content.
    Returns clean text, stripped of HTML tags.
    """
    try:
        # Validate URL format
        if not url.startswith(("http://", "https://")):
            return json.dumps({"error": "URL must start with http:// or https://"})

        response = requests.get(
            url,
            timeout=15,
            headers={
                "User-Agent": "ISOM260-Agent/1.0 (Suffolk University classroom use)"
            }
        )

        response.raise_for_status()

        # Parse HTML and extract text
        soup = BeautifulSoup(response.text, "html.parser")

        # Remove script and style elements
        for script in soup(["script", "style", "nav", "footer", "header"]):
            script.decompose()

        # Get text content
        text = soup.get_text(separator="\n", strip=True)

        # Truncate if too long (LLMs have context limits)
        if len(text) > 3000:
            text = text[:3000] + "\n\n[... content truncated for length ...]"

        return json.dumps({
            "url": url,
            "content": text,
            "length": len(text)
        })

    except requests.exceptions.Timeout:
        return json.dumps({"error": f"Request to {url} timed out after 15 seconds."})
    except requests.exceptions.ConnectionError:
        return json.dumps({"error": f"Could not connect to {url}. The site may be down or the URL may be wrong."})
    except requests.exceptions.HTTPError as e:
        return json.dumps({"error": f"HTTP error {e.response.status_code} when fetching {url}."})
    except requests.exceptions.RequestException as e:
        return json.dumps({"error": f"Failed to fetch {url}: {str(e)}"})

print("📝 Tool defined: web_fetch")
print("   🕸️ Can fetch and read any public web page!")

In [ ]:
# ============================================
# Register all 3 tools — our full toolkit!
# ============================================

all_tools = [calculator_tool, wikipedia_tool, web_fetch_tool]
all_functions = {
    "calculator": calculator,
    "wikipedia_search": wikipedia_search,
    "web_fetch": web_fetch
}

print("✅ 3 tools registered:")
for t in all_tools:
    print(f"   🛠️  {t.name}")
print("\n🎯 Our agent now has: math + Wikipedia + web browsing!")

In [ ]:
# ============================================
# 🎯 Multi-tool test: Wikipedia + Calculator
# ============================================

run_agent(
    "Look up Suffolk University on Wikipedia and tell me when it was founded. "
    "Then calculate how many years it has been operating as of 2026.",
    all_tools, all_functions
)

### 🌟 Key Insight: Real vs. Simulated

Notice the difference from Session 4:

| Session 4 (Simulated) | Session 5 (Real) |
|---|---|
| `stocks = {"AAPL": {"price": 242.58}}` | `requests.get("https://en.wikipedia.org/...")` |
| Always returns the same data | Returns live, current data |
| Never fails | Can timeout, return 404, or error |
| Limited to what you hardcoded | Can look up **anything** |

**The agent loop is identical.** The only thing that changed is what's inside the tool functions. That's the beauty of the pattern — you can swap in any tool without changing the agent architecture.

---

## 🛡️ Part 5: When Things Go Wrong (Error Handling)

Real APIs fail. Pages don't exist. Servers time out. Networks go down.

A production agent needs to **handle errors gracefully** — not crash, not hallucinate, but tell the user what went wrong and try alternatives.

Let's test how our agent handles failures.

In [ ]:
# ============================================
# Test 1: Wikipedia page that doesn't exist
# ============================================

run_agent(
    "Look up 'Xyzzy_Nonexistent_Page_12345' on Wikipedia.",
    all_tools, all_functions
)

In [ ]:
# ============================================
# Test 2: Website that doesn't exist
# ============================================

run_agent(
    "Fetch the web page at https://thiswebsitedoesnotexist12345.com",
    all_tools, all_functions
)

In [ ]:
# ============================================
# 🎯 YOUR TURN: Test 3 more edge cases!
# ============================================
# Try these or come up with your own:

# Edge case 1: Division by zero
# run_agent("What is 100 divided by 0?", all_tools, all_functions)

# Edge case 2: A Wikipedia topic with special characters
# run_agent("Look up 'C++' on Wikipedia.", all_tools, all_functions)

# Edge case 3: A question that needs a tool but might not have an answer
# run_agent("What is the population of the city of Atlantis?", all_tools, all_functions)

# Try your own edge case below:
# run_agent("YOUR QUESTION HERE", all_tools, all_functions)

### 🌟 Key Insight: Error Handling

Did you notice how the agent handled errors?

1. **The tool returned an error message** (not a Python exception)
2. **The agent read the error** and explained it to the user
3. **No crash** — the agent kept running and gave a helpful response

This is a critical pattern in production agents:
- ✅ Tools should **catch exceptions** and return error strings
- ✅ The LLM should **interpret errors** and communicate them clearly
- ❌ Tools should NOT let exceptions bubble up and crash the agent
- ❌ The LLM should NOT pretend the tool succeeded when it failed

In NanoClaw, every single tool has try/except blocks. The agent never crashes — it always explains what went wrong.

---

## 💾 Part 6: Agent Memory — Conversations That Remember

So far, every call to `run_agent()` starts fresh. The agent has **no memory** of previous questions. Watch:

In [ ]:
# ============================================
# The Problem: No Memory!
# ============================================

# Conversation 1
run_agent("Who founded Tesla Motors?", all_tools, all_functions)

# Conversation 2 — the agent has NO IDEA what "he" refers to!
run_agent("How old is he now in 2026?", all_tools, all_functions)

In [ ]:
# ============================================
# 💾 Agent with Memory — the upgrade!
# ============================================
# The trick: instead of starting a fresh conversation each time,
# we keep the FULL conversation history and pass it to Gemini.

def run_agent_with_memory(user_message: str, tool_declarations: list, tool_functions: dict,
                          conversation_history: list = None, verbose=True):
    """
    Run an AI agent that REMEMBERS previous conversations.

    The secret? We just keep appending to the same conversation_history list.
    Gemini sees all previous messages and can refer back to them.
    """
    if conversation_history is None:
        conversation_history = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"🗣️  User: {user_message}")
        print(f"   📝 Memory: {len(conversation_history)} previous messages")
        print(f"{'='*60}")

    # Add user message to history
    conversation_history.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    config = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=tool_declarations)],
        system_instruction="You are a helpful assistant. Use the available tools whenever they would help. Remember previous messages in our conversation."
    )

    step = 0

    while True:
        step += 1

        # 📡 Send FULL history to Gemini — this IS the memory!
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            config=config,
            contents=conversation_history,  # <-- FULL history = memory!
        )

        if verbose:
            print(f"\n🔄 Step {step}")

        model_content = response.candidates[0].content
        conversation_history.append(model_content)

        # Check for function calls
        function_calls = [p for p in model_content.parts if p.function_call]

        if not function_calls:
            # 🎉 Done — extract final answer
            final_answer = response.text
            if verbose:
                print(f"\n🤖 Agent Answer:\n{final_answer}")
                print(f"\n✅ Done in {step} step(s) | Memory: {len(conversation_history)} messages")
            return final_answer, conversation_history

        # Print any thinking text
        for part in model_content.parts:
            if not part.function_call and part.text:
                if verbose:
                    text = part.text
                    print(f"   🧠 Thinking: {text[:150]}..." if len(text) > 150 else f"   🧠 Thinking: {text}")

        # Execute function calls
        function_response_parts = []
        for fc_part in function_calls:
            tool_name = fc_part.function_call.name
            tool_args = dict(fc_part.function_call.args)

            if verbose:
                print(f"   🛠️  Calling tool: {tool_name}")
                print(f"      Input: {json.dumps(tool_args, default=str)}")

            if tool_name in tool_functions:
                result = tool_functions[tool_name](**tool_args)
            else:
                result = f"Error: Unknown tool '{tool_name}'"

            if verbose:
                print(f"      Result: {result[:200]}..." if len(result) > 200 else f"      Result: {result}")

            function_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={"result": result}
                )
            )

        # Add function results to history
        conversation_history.append(
            types.Content(role="user", parts=function_response_parts)
        )

        # Safety: prevent infinite loops
        if step > 10:
            return "Error: Agent exceeded maximum steps.", conversation_history

print("✅ Agent with memory defined!")

In [ ]:
# ============================================
# 🎯 Memory in action: Multi-turn conversation
# ============================================
# Watch how the agent remembers previous answers!

history = []

# Turn 1: Ask about a person
answer, history = run_agent_with_memory(
    "Who founded Tesla Motors?",
    all_tools, all_functions, history
)

# Turn 2: Reference "he" — agent must remember who we're talking about
answer, history = run_agent_with_memory(
    "How old is he now in 2026?",
    all_tools, all_functions, history
)

# Turn 3: Build on BOTH previous answers
answer, history = run_agent_with_memory(
    "If he started the company at that age, how many years has he been running it?",
    all_tools, all_functions, history
)

In [ ]:
# ============================================
# 🔍 Peek inside the memory
# ============================================
# Memory isn't magic — it's literally just a Python list!

print(f"📝 Total messages in memory: {len(history)}\n")

for i, msg in enumerate(history):
    role = msg.role
    parts_preview = []
    for p in msg.parts:
        if p.text:
            parts_preview.append(p.text[:60])
        elif p.function_call:
            parts_preview.append(f"call:{p.function_call.name}")
        elif p.function_response:
            parts_preview.append(f"result:{p.function_response.name}")
    print(f"   {i+1}. [{role:>5}] {' | '.join(parts_preview)}...")

print(f"\n💡 That's it! Memory = a Python list. Nothing magical.")

### 🌟 Key Insight: How Memory Works

Agent memory is surprisingly simple:

```python
# Without memory (Session 4):
contents = [new_message]              # Only the current message

# With memory (Session 5):
contents = history + [new_message]     # ALL previous messages + current
```

That's it. **Memory = keeping the conversation history and sending it all back to the LLM.**

In production, there are fancier approaches:
- **Sliding window** — only keep the last N messages (saves tokens/cost)
- **Summarization** — periodically summarize old messages into a shorter form
- **Database storage** — save conversations to a database for persistence across sessions
- **RAG** — retrieve only relevant past conversations

But the core idea is always the same: give the LLM access to previous context.

---

## 🏆 Part 7: Challenge — Put It All Together

You now have an agent with **real APIs**, **error handling**, and **memory**. Let's test it with a complex, multi-step question that uses everything.

In [ ]:
# ============================================
# 🏆 Complex Challenge: Multi-tool + Memory
# ============================================
# This question requires Wikipedia + Calculator + Memory

history = []

# Step 1: Research
answer, history = run_agent_with_memory(
    "I'm researching universities in Boston. Look up Suffolk University on Wikipedia "
    "and tell me about it.",
    all_tools, all_functions, history
)

# Step 2: Compare (requires memory of step 1)
answer, history = run_agent_with_memory(
    "Now look up Harvard University. How much older is Harvard than Suffolk?",
    all_tools, all_functions, history
)

# Step 3: Synthesize (requires memory of both steps)
answer, history = run_agent_with_memory(
    "Based on what you've learned about both universities, which one might be better "
    "for someone interested in business and technology? Use the information from our conversation.",
    all_tools, all_functions, history
)

In [ ]:
# ============================================
# 🎨 BUILD YOUR OWN REAL TOOL
# ============================================
# Now it's your turn! Create a tool that calls a REAL API.
#
# Some free APIs that don't need API keys:
#   - Open Trivia: https://opentdb.com/api.php?amount=1
#   - Dog Facts: https://dogapi.dog/api/v2/facts
#   - Numbers API: http://numbersapi.com/42
#   - Bored API: https://bored-api.appbrewery.com/random
#   - JSON Placeholder: https://jsonplaceholder.typicode.com/posts/1
#
# Template:

my_tool = types.FunctionDeclaration(
    name="your_tool_name",                # <-- Change this
    description=(
        "Describe what your tool does. "   # <-- Change this
        "Be detailed so Gemini knows when to use it."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "param1": {                    # <-- Change parameter name
                "type": "string",
                "description": "What this parameter means"
            }
        },
        "required": ["param1"]
    }
)

def your_tool_name(param1: str) -> str:    # <-- Match the tool name!
    """Your tool logic here."""
    try:
        # Call a real API here!
        response = requests.get(f"https://some-api.com/{param1}", timeout=10)
        response.raise_for_status()
        data = response.json()
        return json.dumps(data)
    except requests.exceptions.RequestException as e:
        return json.dumps({"error": str(e)})

# Register all tools including yours
# my_tools = [calculator_tool, wikipedia_tool, web_fetch_tool, my_tool]
# my_functions = {
#     "calculator": calculator,
#     "wikipedia_search": wikipedia_search,
#     "web_fetch": web_fetch,
#     "your_tool_name": your_tool_name
# }

# Test it!
# run_agent("Your test question here", my_tools, my_functions)

---

## 💭 Reflection — What You Built Today

### Session 4 vs. Session 5

| Feature | Session 4 | Session 5 |
|---|---|---|
| **Data** | Hardcoded dictionaries | Live Wikipedia & web APIs |
| **Error handling** | None (tools always work) | Try/except, graceful error messages |
| **Memory** | None (each call is fresh) | Full conversation history |
| **Tools** | Calculator, fake stocks, fake company info | Calculator, Wikipedia, web fetcher |
| **Realism** | Classroom demo | Getting close to production |

### The Production Gap

What's still missing vs. NanoClaw?

| What You Built | What NanoClaw Does |
|---|---|
| 3 real tools | Dozens of tools (email, calendar, files, code execution) |
| In-memory conversation history | Persistent database storage |
| Runs in Colab | Runs in isolated Docker containers 24/7 |
| You type messages | Connected to WhatsApp, web UI, Slack |
| No security | Sandboxed execution, permission system |
| ~50 lines of agent code | ~500 lines of core code |

But the **core architecture is identical**:

```
1. Receive message
2. Send to LLM with tool definitions + conversation history
3. If LLM wants to use a tool → execute it → send result back
4. Repeat until LLM has a final answer
5. Store conversation in history (memory)
6. Return answer to user
```

### 💡 The Business Insight

The jump from Session 4 to Session 5 is exactly the jump every AI startup makes:
1. **Prototype** → hardcoded data, no error handling, no memory
2. **MVP** → real APIs, basic error handling, session memory
3. **Production** → robust infrastructure, security, persistence

You've gone from prototype to MVP. Session 8 covers the production leap.

---

## 🚀 What's Next?

- **Session 6**: Computer Vision — teaching AI to see and understand images
- **Session 8**: AI Agents & Automation — NanoClaw architecture, production patterns, building your own agent product
- **Challenge**: Add a 4th real-API tool to this notebook (weather, news, trivia, etc.)
- **Think about**: What tool would make YOUR daily workflow easier? That's a product idea.

---

**ISOM 260: AI for Business** | Suffolk University | Session 5

🌐 [isom-260.vercel.app](https://isom-260.vercel.app)